# This is for evaluating the sft (not aligned) models - QWEN 2.5

## Load models first

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "unsloth/gpt-oss-20b-unsloth-bnb-4bit"
ADAPTER_DIR = "runs/oss20b_unsloth_qlora/oss20b_unsloth_qlora/adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
).eval()

adapter_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()

def chat_generate(model, prompt, max_new_tokens=256, temperature=0.2, top_p=0.95):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        top_p=top_p,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=False)


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 6/6 [00:32<00:00,  5.35s/it]


## Get model responses - for human eval

In [2]:
prompts = [
    "Write a Python function to deduplicate a list while preserving order.",
    "Explain why look-ahead bias is dangerous in backtesting, and how to avoid it. Give a short example.",
    "Given a list of dicts with keys {id, email}, write code to drop near-duplicate emails. Be careful about case/whitespace.",
]

for p in prompts:
    print("\n" + "="*100)
    print("PROMPT:", p)
    print("="*100)

    print("\n[BASE]\n")
    print(chat_generate(base_model, p, max_new_tokens=256, temperature=0.2))

    print("\n" + "-"*100)

    print("\n[ADAPTER]\n")
    print(chat_generate(adapter_model, p, max_new_tokens=256, temperature=0.2))



PROMPT: Write a Python function to deduplicate a list while preserving order.

[BASE]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a Python function to deduplicate a list while preserving order.<|im_end|>
<|im_start|>assistant
def deduplicate_list(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result<|im_end|>

----------------------------------------------------------------------------------------------------

[ADAPTER]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a Python function to deduplicate a list while preserving order.<|im_end|>
<|im_start|>assistant
def deduplicate_list(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            result.append(item)
            seen.add(item

## Load eval model (gpt)

In [6]:
!pip install -q python-dotenv


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env into os.environ

assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not found in .env"
print("OPENAI_API_KEY loaded from .env ✅")



OPENAI_API_KEY loaded from .env ✅


In [8]:
from openai import OpenAI

client = OpenAI()  # automatically reads OPENAI_API_KEY from env


In [9]:
import json

def evaluate_pair_gpt4o(prompt, base_answer, adapter_answer):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert software engineer evaluating two answers. "
                    "Be strict and technical. Respond ONLY in valid JSON."
                ),
            },
            {
                "role": "user",
                "content": f"""
PROMPT:
{prompt}

ANSWER_A (BASE):
{base_answer}

ANSWER_B (ADAPTER):
{adapter_answer}

Return JSON exactly like:
{{
  "winner": "A" | "B" | "tie",
  "justification": "short technical explanation"
}}
""".strip(),
            },
        ],
    )

    return json.loads(response.choices[0].message.content)


## Pilot test

In [10]:
prompt = "Write a Python function to deduplicate a list while preserving order."

base_out = chat_generate(base_model, prompt, max_new_tokens=200, temperature=0.2)
adapter_out = chat_generate(adapter_model, prompt, max_new_tokens=200, temperature=0.2)

result = evaluate_pair_gpt4o(prompt, base_out, adapter_out)

print(result)


{'winner': 'tie', 'justification': 'Both answers provide the same correct implementation of the deduplication function, preserving order and using a set for efficient membership checking.'}


## AI Eval using GPT

In [15]:
from openai import OpenAI
import json
import re

# -----------------------------
# Config
# -----------------------------
client = OpenAI()  # uses OPENAI_API_KEY from env

prompts = [
    "Write a Python function to deduplicate a list while preserving order.",
    "Explain why look-ahead bias is dangerous in backtesting, and how to avoid it. Give a short example.",
    "Given a list of dicts with keys {id, email}, write code to drop near-duplicate emails. Be careful about case/whitespace.",
    "What is the difference between list and tuple in Python, and when would you use each?",
    "Write a Python function that safely parses an integer from a string and handles errors gracefully.",
]

# -----------------------------
# Helper: clean chat tokens
# -----------------------------
def strip_chat_tokens(text: str) -> str:
    for tok in ["<|im_start|>", "<|im_end|>"]:
        text = text.replace(tok, "")
    return text.strip()

# -----------------------------
# GPT-4o-mini evaluator
# -----------------------------
def evaluate_pair(prompt, base_answer, adapter_answer):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,  # deterministic judge
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert software engineer judging two answers. "
                    "Be strict and technical. Respond ONLY in valid JSON."
                ),
            },
            {
                "role": "user",
                "content": f"""
PROMPT:
{prompt}

ANSWER_A (BASE):
{base_answer}

ANSWER_B (ADAPTER):
{adapter_answer}

Return JSON exactly like:
{{
  "winner": "A" | "B" | "tie",
  "justification": "short technical explanation"
}}
""".strip(),
            },
        ],
    )

    text = response.choices[0].message.content
    match = re.search(r"\{.*\}", text, re.S)

    if not match:
        raise ValueError("Evaluator did not return valid JSON:\n" + text)

    return json.loads(match.group())

# -----------------------------
# Generate + Evaluate
# -----------------------------
results = []

for i, p in enumerate(prompts, 1):
    base_out = strip_chat_tokens(
        chat_generate(base_model, p, max_new_tokens=256, temperature=0.2)
    )
    adapter_out = strip_chat_tokens(
        chat_generate(adapter_model, p, max_new_tokens=256, temperature=0.2)
    )

    eval_result = evaluate_pair(p, base_out, adapter_out)

    results.append({
        "prompt": p,
        "base": base_out,
        "adapter": adapter_out,
        "evaluation": eval_result,
    })

    print("\n" + "=" * 120)
    print(f"PROMPT {i}: {p}")
    print("-" * 120)
    print("[BASE]\n", base_out)
    print("\n" + "-" * 120)
    print("[ADAPTER]\n", adapter_out)
    print("\n" + "-" * 120)
    print("EVAL RESULT:", eval_result)

# -----------------------------
# Optional: save results
# -----------------------------
with open("runs/gpt4o_eval_5prompts.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print("\n✅ Saved evaluation results to runs/gpt4o_eval_5prompts.jsonl")




PROMPT 1: Write a Python function to deduplicate a list while preserving order.
------------------------------------------------------------------------------------------------------------------------
[BASE]
 system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python function to deduplicate a list while preserving order.
assistant
def deduplicate_list(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

------------------------------------------------------------------------------------------------------------------------
[ADAPTER]
 system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python function to deduplicate a list while preserving order.
assistant
def deduplicate_list(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
           